# MetadataBuilder Tutorial

This tutorial demonstrates how to use the `MetadataBuilder` to create standardized metadata tables across heterogeneous datasets using optional alias mappings.

## Overview

The `MetadataBuilder` solves a common problem when working with multiple genomics datasets: **factor levels are named inconsistently across datasets**. For example, carbon sources might be labeled as "D-glucose", "dextrose", "glucose", etc.

**Key difference from filtering**: This system does NOT filter or exclude any data. Instead, it normalizes factor level names to create standardized metadata views. ALL data is included, just with standardized names.

You specify:

1. **Factor aliases** (optional): Mappings to normalize factor level names (e.g., `glucose: ["D-glucose", "dextrose"]`)
2. **Property mappings**: Where to find each property in each dataset (e.g., repo-level vs field-level)

## Key Features

- **No filtering**: ALL data is included with normalized names
- **Optional aliases**: If no alias is configured, original values pass through unchanged
- **Case-insensitive matching**: "D-glucose" matches "d-glucose"
- **Three output modes**:
  - `conditions`: Just normalized metadata (no data retrieval)
  - `samples`: Sample-level metadata (one row per sample_id)
  - `full_data`: All measurements with normalized metadata
- **External configuration**: No hardcoded dataset logic in your analysis code

## Setup

In [1]:
from pathlib import Path
import yaml
import pandas as pd
from tfbpapi.datainfo.metadata_builder import MetadataBuilder, normalize_value
from tfbpapi.datainfo import DataCard

/home/chase/code/tfbp/tfbpapi/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Understanding Dataset Structures

Before normalizing, let's examine how experimental conditions are structured in two representative datasets.

### Repo-Level Conditions: Kemmeren 2014

All samples in this dataset share the same experimental conditions defined at the repository level.

In [2]:
# Load Kemmeren 2014 datacard
kemmeren_card = DataCard("BrentLab/kemmeren_2014")

# Get repo-level experimental conditions
exp_conds = kemmeren_card.get_experimental_conditions("kemmeren_2014")

print("Kemmeren 2014 - Repo-Level Experimental Conditions")
print("=" * 60)
print(f"Temperature: {exp_conds.get('temperature_celsius')}°C")
print(f"Media: {exp_conds.get('media', {}).get('name')}")

carbon_source = exp_conds.get('media', {}).get('carbon_source', [])
if carbon_source:
    compounds = [cs.get('compound') for cs in carbon_source]
    print(f"Carbon source: {compounds}")

print("\nAll samples in this dataset share these conditions.")

Kemmeren 2014 - Repo-Level Experimental Conditions
Temperature: 30°C
Media: synthetic_complete
Carbon source: ['D-glucose']

All samples in this dataset share these conditions.


### Field-Level Conditions: Harbison 2004

This dataset has different experimental conditions for different samples, defined in the `condition` field's definitions.

In [3]:
# Load Harbison 2004 datacard
harbison_card = DataCard("BrentLab/harbison_2004")

# Get a summary of condition field levels
print(harbison_card.summarize_field_levels(
    "harbison_2004",
    "condition",
    properties=["media.name", "media.carbon_source"],
    max_levels=6
))

Field: condition
Levels: 14

YPD:
  media.name: YPD
  media.carbon_source: [{'compound': 'D-glucose', 'concentration_percent': 2}]

SM:
  media.name: synthetic_complete
  media.carbon_source: None

RAPA:
  media.name: YPD
  media.carbon_source: [{'compound': 'D-glucose', 'concentration_percent': 2}]

H2O2Hi:
  media.name: YPD
  media.carbon_source: [{'compound': 'D-glucose', 'concentration_percent': 2}]

H2O2Lo:
  media.name: YPD
  media.carbon_source: [{'compound': 'D-glucose', 'concentration_percent': 2}]

Acid:
  media.name: YPD
  media.carbon_source: [{'compound': 'D-glucose', 'concentration_percent': 2}]

... and 8 more levels


## Example 8: Extracting Properties from Field Definitions

The MetadataBuilder can extract properties from field definitions and add them as new columns. This is useful for creating analysis-ready metadata tables.

For harbison_2004, the base metadata has `condition` as a field, but we want to extract:
- `growth_media` from `media.name`
- `carbon_source` from `media.carbon_source.compound`

And normalize carbon source names using aliases.

In [4]:
# Configuration with property extraction and aliases
extraction_config = {
    "factor_aliases": {
        "carbon_source": {
            "glucose": ["D-glucose", "Glucose", "dextrose", "Dextrose"],
            "galactose": ["D-galactose", "Galactose"],
            "raffinose": ["D-raffinose", "Raffinose"]
        }
    },
    "BrentLab/harbison_2004": {
        "dataset": {
            "harbison_2004": {
                "growth_media": {
                    "field": "condition",
                    "path": "media.name"
                },
                "carbon_source": {
                    "field": "condition",
                    "path": "media.carbon_source.compound"
                }
            }
        }
    }
}

config_path_extraction = Path("/tmp/metadata_extraction.yaml")
with open(config_path_extraction, 'w') as f:
    yaml.dump(extraction_config, f)

# Build metadata with extraction
builder_extraction = MetadataBuilder(config_path_extraction)
results_extraction = builder_extraction.build_metadata(
    repos=[("BrentLab/harbison_2004", "harbison_2004")],
    mode="samples"
)

df_extracted = results_extraction["BrentLab/harbison_2004"]["data"]

df_extracted

# print(f"Columns: {list(df_extracted.columns)}")
# print(f"\nShape: {df_extracted.shape}")
# print("\nSample data showing base + extracted columns:")
# print(df_extracted[['sample_id', 'condition', 'growth_media', 'carbon_source']].head(10))

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00,  2.88it/s]


,sample_id,regulator_locus_tag,regulator_symbol,condition,carbon_source,growth_media
0,1,YSC0017,MATA1,YPD,glucose,YPD
1,2,YAL051W,OAF1,YPD,glucose,YPD
2,3,YBL005W,PDR3,YPD,glucose,YPD
3,4,YBL008W,HIR1,YPD,glucose,YPD
4,5,YBL021C,HAP3,YPD,glucose,YPD
...,...,...,...,...,...,...
347,348,YPR104C,FHL1,YPD,glucose,YPD
348,349,YPR196W,YPR196W,YPD,glucose,YPD
349,350,YPR199C,ARR1,H2O2Hi,glucose,YPD
350,351,YPR199C,ARR1,YPD,glucose,YPD
